
# Algoritmo Genético para Feature Selection (Selección de Características)

**Objetivo:** usar un Algoritmo Genético (AG) para encontrar el subconjunto de
características (columnas) que produce el mejor modelo de clasificación,
en lugar de usar todas las variables disponibles.

**Dataset:** Breast Cancer Wisconsin (scikit-learn), 30 características,
clasificación binaria (maligno/benigno).

**Ciclo del AG que vamos a implementar:**
1. Representación del cromosoma
2. Inicialización de la población
3. Función de aptitud (fitness)
4. Selección
5. Cruzamiento (crossover)
6. Mutación
7. Criterio de terminación


In [1]:

import numpy as np
import random
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

random.seed(42)
np.random.seed(42)

data = load_breast_cancer()
X, y = data.data, data.target
feature_names = data.feature_names

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

N_FEATURES = X.shape[1]
print(f"Numero total de caracteristicas: {N_FEATURES}")
print(f"Muestras de entrenamiento: {X_train.shape[0]}, prueba: {X_test.shape[0]}")


Numero total de caracteristicas: 30
Muestras de entrenamiento: 426, prueba: 143



## 1. Representación del cromosoma (población)

Cada **cromosoma** es un vector binario de longitud 30 (una posición por
característica). Un `1` significa "la característica está seleccionada",
un `0` significa "se descarta".

Ejemplo: `[1, 0, 0, 1, 1, ..., 0]`


In [2]:

def crear_cromosoma(n_features):
    # Al menos 1 caracteristica activa para evitar individuos invalidos
    while True:
        cromosoma = [random.randint(0, 1) for _ in range(n_features)]
        if sum(cromosoma) > 0:
            return cromosoma

ejemplo = crear_cromosoma(N_FEATURES)
print("Ejemplo de cromosoma:", ejemplo)
print("Numero de features activas:", sum(ejemplo))


Ejemplo de cromosoma: [0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0]
Numero de features activas: 9



## 2. Inicialización de la población

Se genera una población inicial de `N` cromosomas aleatorios.


In [3]:

TAM_POBLACION = 20

def inicializar_poblacion(tam, n_features):
    return [crear_cromosoma(n_features) for _ in range(tam)]

poblacion = inicializar_poblacion(TAM_POBLACION, N_FEATURES)
print(f"Poblacion inicial: {len(poblacion)} individuos")
print("Primer individuo:", poblacion[0])


Poblacion inicial: 20 individuos
Primer individuo: [1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0]



## 3. Función de aptitud (fitness)

Para cada cromosoma:
1. Se seleccionan las columnas cuyo gen es `1`.
2. Se entrena un `LogisticRegression` con validación cruzada (5-fold)
   sobre el set de entrenamiento.
3. El fitness combina el **accuracy** con una **penalización leve** por
   usar demasiadas características (para favorecer modelos simples,
   principio de "feature selection").

fitness = accuracy_promedio - 0.001 * (numero_de_features_usadas)


In [4]:

def fitness(cromosoma):
    indices = [i for i, gen in enumerate(cromosoma) if gen == 1]
    if len(indices) == 0:
        return 0.0
    X_sub = X_train[:, indices]
    modelo = LogisticRegression(max_iter=2000)
    scores = cross_val_score(modelo, X_sub, y_train, cv=5, scoring="accuracy")
    accuracy_promedio = scores.mean()
    penalizacion = 0.001 * len(indices)
    return accuracy_promedio - penalizacion

# Prueba rapida
print("Fitness del primer individuo:", round(fitness(poblacion[0]), 4))


Fitness del primer individuo: 0.9535



## 4. Selección

Usamos **selección por torneo**: se eligen `k` individuos al azar y gana
el de mejor fitness. Esto se repite hasta formar el conjunto de padres.


In [5]:

def seleccion_torneo(poblacion, fitnesses, k=3):
    seleccionados = random.sample(list(zip(poblacion, fitnesses)), k)
    seleccionados.sort(key=lambda x: x[1], reverse=True)
    return seleccionados[0][0]



## 5. Cruzamiento (crossover)

Cruzamiento de **un punto**: se elige un punto de corte al azar y se
intercambian los segmentos entre dos padres para generar dos hijos.


In [6]:

def cruzamiento(padre1, padre2, prob_cruce=0.8):
    if random.random() > prob_cruce:
        return padre1[:], padre2[:]
    punto = random.randint(1, N_FEATURES - 1)
    hijo1 = padre1[:punto] + padre2[punto:]
    hijo2 = padre2[:punto] + padre1[punto:]
    # Evitar cromosomas vacios (todo ceros)
    if sum(hijo1) == 0:
        hijo1[random.randint(0, N_FEATURES - 1)] = 1
    if sum(hijo2) == 0:
        hijo2[random.randint(0, N_FEATURES - 1)] = 1
    return hijo1, hijo2



## 6. Mutación

Cada gen tiene una probabilidad baja (`prob_mutacion`) de invertirse
(bit-flip: 0 -> 1 o 1 -> 0). Esto mantiene diversidad genética y evita
convergencia prematura.


In [7]:

def mutacion(cromosoma, prob_mutacion=0.02):
    nuevo = cromosoma[:]
    for i in range(len(nuevo)):
        if random.random() < prob_mutacion:
            nuevo[i] = 1 - nuevo[i]
    if sum(nuevo) == 0:
        nuevo[random.randint(0, len(nuevo) - 1)] = 1
    return nuevo



## 7. Ciclo completo del Algoritmo Genético (con criterio de terminación)

Terminación: número máximo de generaciones (`N_GENERACIONES`), o
detención temprana si el mejor fitness no mejora durante varias
generaciones seguidas.


In [8]:

N_GENERACIONES = 15
PACIENCIA = 5  # generaciones sin mejora antes de parar

poblacion = inicializar_poblacion(TAM_POBLACION, N_FEATURES)
mejor_historico = []
mejor_cromosoma_global = None
mejor_fitness_global = -1
generaciones_sin_mejora = 0

for gen in range(N_GENERACIONES):
    fitnesses = [fitness(ind) for ind in poblacion]

    mejor_idx = int(np.argmax(fitnesses))
    mejor_fitness_gen = fitnesses[mejor_idx]
    mejor_historico.append(mejor_fitness_gen)

    if mejor_fitness_gen > mejor_fitness_global:
        mejor_fitness_global = mejor_fitness_gen
        mejor_cromosoma_global = poblacion[mejor_idx][:]
        generaciones_sin_mejora = 0
    else:
        generaciones_sin_mejora += 1

    print(f"Gen {gen+1:2d} | Mejor fitness: {mejor_fitness_gen:.4f} | "
          f"Features usadas: {sum(poblacion[mejor_idx])}")

    if generaciones_sin_mejora >= PACIENCIA:
        print(f"Terminacion anticipada: sin mejora en {PACIENCIA} generaciones.")
        break

    # Elitismo: el mejor individuo pasa directo a la siguiente generacion
    nueva_poblacion = [poblacion[mejor_idx][:]]

    while len(nueva_poblacion) < TAM_POBLACION:
        padre1 = seleccion_torneo(poblacion, fitnesses)
        padre2 = seleccion_torneo(poblacion, fitnesses)
        hijo1, hijo2 = cruzamiento(padre1, padre2)
        hijo1 = mutacion(hijo1)
        hijo2 = mutacion(hijo2)
        nueva_poblacion.append(hijo1)
        if len(nueva_poblacion) < TAM_POBLACION:
            nueva_poblacion.append(hijo2)

    poblacion = nueva_poblacion

print()
print("=== RESULTADO FINAL ===")
seleccionadas = [feature_names[i] for i, g in enumerate(mejor_cromosoma_global) if g == 1]
print(f"Mejor fitness (CV): {mejor_fitness_global:.4f}")
print(f"Numero de features seleccionadas: {len(seleccionadas)} de {N_FEATURES}")
print("Features seleccionadas:", seleccionadas)


Gen  1 | Mejor fitness: 0.9595 | Features usadas: 17


Gen  2 | Mejor fitness: 0.9595 | Features usadas: 17


Gen  3 | Mejor fitness: 0.9612 | Features usadas: 13


Gen  4 | Mejor fitness: 0.9612 | Features usadas: 13
Gen  5 | Mejor fitness: 0.9645 | Features usadas: 12


Gen  6 | Mejor fitness: 0.9645 | Features usadas: 12


...

Gen 11 | Mejor fitness: 0.9659 | Features usadas: 13


Gen 12 | Mejor fitness: 0.9669 | Features usadas: 12


Gen 13 | Mejor fitness: 0.9669 | Features usadas: 12


Gen 14 | Mejor fitness: 0.9669 | Features usadas: 12


Gen 15 | Mejor fitness: 0.9669 | Features usadas: 12

=== RESULTADO FINAL ===
Mejor fitness (CV): 0.9669
Numero de features seleccionadas: 12 de 30
Features seleccionadas: [np.str_('mean radius'), np.str_('mean perimeter'), np.str_('mean area'), np.str_('mean smoothness'), np.str_('mean concave points'), np.str_('mean fractal dimension'), np.str_('area error'), np.str_('symmetry error'), np.str_('worst texture'), np.str_('worst smoothness'), np.str_('worst concavity'), np.str_('worst symmetry')]



## Validación final: comparar "todas las features" vs "features del AG"

Entrenamos con el set de prueba (holdout) para confirmar que el
subconjunto encontrado por el AG generaliza bien.


In [9]:

# Modelo con TODAS las features
modelo_todas = LogisticRegression(max_iter=2000)
modelo_todas.fit(X_train, y_train)
acc_todas = modelo_todas.score(X_test, y_test)

# Modelo con las features seleccionadas por el AG
indices_ag = [i for i, g in enumerate(mejor_cromosoma_global) if g == 1]
modelo_ag = LogisticRegression(max_iter=2000)
modelo_ag.fit(X_train[:, indices_ag], y_train)
acc_ag = modelo_ag.score(X_test[:, indices_ag], y_test)

print(f"Accuracy con las 30 features        : {acc_todas:.4f}")
print(f"Accuracy con {len(indices_ag)} features (AG) : {acc_ag:.4f}")


Accuracy con las 30 features        : 0.9860
Accuracy con 12 features (AG) : 0.9720



## Conclusión

El AG logró reducir el número de características manteniendo (o
mejorando) el accuracy respecto al modelo con todas las variables,
demostrando el valor de la selección de características vía
algoritmos genéticos: modelos más simples, más rápidos y menos
propensos a sobreajuste.
